# Compare two opsim databases

- **author :** Sylvie Dagoret-Campagne
- **affliliation :** IJCLab/IN2P3/CNRS
- **creation date :** 2026-07-24
- **last date :** 2026-07-24

Compares two Rubin/LSST cadence simulations (opsim databases), side by side:
- categorical columns: stacked horizontal bar plots by band, the two simulations shown as two horizontal subplots on the same figure
- numerical columns: value vs MJD, the two simulations shown as two vertically-stacked subplots sharing the same x (MJD) axis, with a secondary date (YYYY-MM-DD) axis on top of the first subplot
- adds `raDeg` (0-360 deg) / `decDeg` (-90 to +90 deg) columns built from the RA/Dec columns found in each dataframe

## Requirements
- Install :
https://github.com/LSSTDESC/OpSimSummaryV2

In [ ]:
import opsimsummaryv2 as op
import os
import sys
import logging
import re

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from astropy.time import Time

In [ ]:
import ipywidgets

%matplotlib widget

import ipywidgets as widgets

widgets.IntSlider()

print("matplotlib:", mpl.__version__)
print("backend:", mpl.get_backend())
print("ipywidgets:", ipywidgets.__version__)

## 1) Logging

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)

if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)

log.info("Logging implemented !")

## 2) Db Configuration

In [ ]:
# Notebook input

# -- Notebook tag ---------------------------------------------------------
NB_TAG = "Compare2OpsimDb_01"
DIR_DATA_IN = "/Users/dagoret/DATA/OpSim"

# -- Output figures --------------------------------------------------------
DIR_FIGS = f"./figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
log.info("Figure directory: %s", DIR_FIGS)

### List of opsim simulations
See the post at : https://community.lsst.org/t/release-of-v5-3-simulations/12032

In [ ]:
# Opsim Rubin-LSST cadence simulations
list_opsims = [
    "baseline_v5.3.0_10yrs.db",
    "faster_templates_v5.3.0_10yrs.db",
    "ddf_one_less_v5.3.2_10yrs.db",
    "roll_mash_v5.3.0_10yrs.db",
    "roll_u5_v5.3.0_10yrs.db",
    "ddf_sd_v5.3.0_10yrs.db",
    "desi_3040_v5.3.0_10yrs.db",
    "baseline_v5.3.0_11yrs.db",
]
for i, name in enumerate(list_opsims):
    print(i, name)

### Choose the two simulations to compare

In [ ]:
# -- Pick the 2 opsim simulations to compare (indices into list_opsims) ---
index_sim1 = 0
index_sim2 = 1

filename_in1 = list_opsims[index_sim1]
filename_in2 = list_opsims[index_sim2]

opsim_path1 = os.path.join(DIR_DATA_IN, filename_in1)
opsim_path2 = os.path.join(DIR_DATA_IN, filename_in2)

log.info("Simulation 1 : %s", filename_in1)
log.info("Simulation 2 : %s", filename_in2)

In [ ]:
def get_tagname(filename_in):
    m = re.match(r"(.+)\.db$", filename_in)
    return m.group(1) if m else filename_in


tagname1 = get_tagname(filename_in1)
tagname2 = get_tagname(filename_in2)
log.info(f"tag1 = {tagname1}")
log.info(f"tag2 = {tagname2}")

## 3) Read the two Opsim Db

In [ ]:
sql_engine1 = op.OpSimSurvey._get_sql_engine(opsim_path1)
sql_engine2 = op.OpSimSurvey._get_sql_engine(opsim_path2)

In [ ]:
df1 = op.OpSimSurvey._get_df_from_sql(sql_engine1)
df2 = op.OpSimSurvey._get_df_from_sql(sql_engine2)

In [ ]:
log.info(f"columns df1 : {list(df1.columns)}")
log.info(f"columns df2 : {list(df2.columns)}")

## 4) Analyse Setup

In [ ]:
BANDS_ORDER = ["u", "g", "r", "i", "z", "y"]

In [ ]:
df1["filter"] = df1["filter"].str.extract(r"([a-zA-Z]+)")
df2["filter"] = df2["filter"].str.extract(r"([a-zA-Z]+)")

## 5) Plots setup

In [ ]:
# Standard Rubin/LSST filter colors
FILTER_COLORS = {
    "u": "#56b4e9",
    "g": "#008060",
    "r": "#ff4000",
    "i": "#850000",
    "z": "#6600cc",
    "y": "#000000",
}
FILTER_ORDER = ["u", "g", "r", "i", "z", "y"]

### 5.1) Helpers

In [ ]:
# -- savefig: PDF + PNG ------------------------------------------------
def savefig(fig, name, dpi=150):
    """Save *fig* as both PDF and PNG under DIR_FIGS."""
    base = os.path.join(DIR_FIGS, name)
    # fig.savefig(base + ".pdf", dpi=dpi, bbox_inches="tight")
    fig.savefig(base + ".png", dpi=dpi, bbox_inches="tight")
    log.info("Saved figure: %s (.pdf/.png)", base)

In [ ]:
def get_year_ticks(mjd_min, mjd_max):
    """
    Compute the MJD of January 1st for every year covered by
    [mjd_min, mjd_max], used for axis ticks and axvlines.
    """
    t_min = Time(mjd_min, format="mjd", scale="utc")
    t_max = Time(mjd_max, format="mjd", scale="utc")

    year_min = int(t_min.strftime("%Y"))
    year_max = int(t_max.strftime("%Y"))

    year_mjds = []
    year_labels = []

    for year in range(year_min, year_max + 2):
        t_jan1 = Time(f"{year}-01-01T00:00:00", format="isot", scale="utc")
        if mjd_min <= t_jan1.mjd <= mjd_max:
            year_mjds.append(t_jan1.mjd)
            year_labels.append(f"{year}-01-01")

    return np.array(year_mjds), year_labels

### 5.2) RA/Dec: add raDeg / decDeg columns
`raDeg` in [0, 360) degrees, `decDeg` in [-90, 90] degrees, built from whichever `*_ra` / `*_dec` (or `*RA` / `*Dec`) columns exist. Values are auto-detected as radians or degrees.

In [ ]:
def find_ra_dec_columns(df):
    """Find the RA/Dec column names in *df*, matching e.g. fieldRA/fieldDec,
    ra/dec, field_ra/field_dec (case-insensitive)."""
    ra_cols = [c for c in df.columns if re.search(r"(?i)(^|_)ra$", c)]
    dec_cols = [c for c in df.columns if re.search(r"(?i)(^|_)dec$", c)]
    if not ra_cols or not dec_cols:
        raise ValueError(f"No RA/Dec columns found. Columns available: {list(df.columns)}")
    return ra_cols[0], dec_cols[0]


def add_radeg_decdeg(df, ra_col=None, dec_col=None):
    """Add raDeg (0-360 deg) and decDeg (-90 to +90 deg) columns to *df*,
    auto-converting from radians if needed."""
    if ra_col is None or dec_col is None:
        ra_col, dec_col = find_ra_dec_columns(df)

    ra = df[ra_col].to_numpy(dtype=float)
    dec = df[dec_col].to_numpy(dtype=float)

    ra_deg = np.degrees(ra) if np.nanmax(np.abs(ra)) <= (2 * np.pi + 0.01) else ra.copy()
    dec_deg = np.degrees(dec) if np.nanmax(np.abs(dec)) <= (np.pi / 2 + 0.01) else dec.copy()

    df["raDeg"] = np.mod(ra_deg, 360.0)
    df["decDeg"] = dec_deg

    log.info("raDeg/decDeg added from columns (%s, %s)", ra_col, dec_col)
    return df


df1 = add_radeg_decdeg(df1)
df2 = add_radeg_decdeg(df2)

### 5.3) Categorical columns: stacked horizontal bar plots, simulation 1 vs simulation 2 (2 horizontal subplots on the same figure)

In [ ]:
def plot_category_stacked_barh_compare(
    df1, df2, column, tag1, tag2, filter_column="filter", max_categories=30, figsize=(16, 8)
):
    """
    Plot value-count horizontal bar charts (stacked by band) for *column* in
    df1 and df2 side by side, as 2 horizontal subplots on the same figure.
    """
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    for ax, df, tag in zip(axes, (df1, df2), (tag1, tag2)):
        counts = pd.crosstab(df[column], df[filter_column])
        bands = [b for b in FILTER_ORDER if b in counts.columns]
        counts = counts[bands]

        totals = counts.sum(axis=1).sort_values(ascending=False)
        counts = counts.loc[totals.index[:max_categories]]
        counts = counts.iloc[::-1]

        left = np.zeros(len(counts))
        for band in bands:
            ax.barh(
                counts.index.astype(str),
                counts[band].values,
                left=left,
                color=FILTER_COLORS[band],
                label=band,
            )
            left = left + counts[band].values

        ax.set_xlabel("Number of observations")
        ax.set_ylabel(column)
        ax.set_title(f"{column} by band ({tag})")
        ax.legend(title="band", loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)

    fig.tight_layout()
    return fig, axes

In [ ]:
# Columns not plotted: the ones already used to define the bands
CATEGORY_SKIP_COLUMNS = {"filter", "band"}

common_columns = [c for c in df1.columns if c in df2.columns]

for column in common_columns:
    if column in CATEGORY_SKIP_COLUMNS:
        continue
    if pd.api.types.is_numeric_dtype(df1[column]) or pd.api.types.is_numeric_dtype(df2[column]):
        continue
    fig, axes = plot_category_stacked_barh_compare(df1, df2, column, tagname1, tagname2)
    figname = f"compare_{tagname1}_vs_{tagname2}_{column}"
    savefig(fig, figname, dpi=150)
    plt.show()

### 5.4) Numerical columns: value vs MJD, simulation 1 vs simulation 2
2 subplots stacked vertically, sharing the same x (MJD) axis, with the YYYY-MM-DD date axis on top of the first (upper) subplot for zooming.

In [ ]:
def plot_column_vs_mjd_compare(
    df1, df2, column, tag1, tag2, mjd_column="observationStartMJD", filter_column="filter", figsize=(12, 8)
):
    """
    Plot *column* vs MJD for df1 (top subplot) and df2 (bottom subplot),
    stacked vertically and sharing the same x (MJD) axis. A secondary top
    axis with readable dates (YYYY-MM-DD) is added above the top subplot.
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, sharex=True)

    for ax, df, tag in zip((ax1, ax2), (df1, df2), (tag1, tag2)):
        mjd = df[mjd_column].values
        values = df[column]
        for band in FILTER_ORDER:
            mask = df[filter_column] == band
            if mask.sum() == 0:
                continue
            ax.scatter(
                mjd[mask.values],
                values[mask].values,
                s=3,
                alpha=0.5,
                color=FILTER_COLORS[band],
                label=band,
            )
        ax.set_ylabel(column)
        ax.set_title(f"{column} vs MJD ({tag})")
        ax.legend(markerscale=5, title="band", loc="center left", fontsize=8, bbox_to_anchor=(1, 0.5))

    ax2.set_xlabel("MJD")

    # Year ticks / gridlines computed over the combined MJD range of both sims
    mjd_all = np.concatenate([df1[mjd_column].values, df2[mjd_column].values])
    mjd_min, mjd_max = np.nanmin(mjd_all), np.nanmax(mjd_all)
    year_mjds, year_labels = get_year_ticks(mjd_min, mjd_max)
    for ax in (ax1, ax2):
        for ymjd in year_mjds:
            ax.axvline(ymjd, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)

    # Secondary top axis with readable dates, on the upper subplot only
    ax1.set_xlim(mjd_min, mjd_max)
    ax_top = ax1.twiny()
    ax_top.set_xlim(ax1.get_xlim())
    ax_top.set_xticks(year_mjds)
    ax_top.set_xticklabels(year_labels, rotation=45, ha="left")
    ax_top.set_xlabel("Date")

    fig.tight_layout()
    return fig, (ax1, ax2)

In [ ]:
# Columns not plotted: the MJD itself, and non-numeric / categorical columns
SKIP_COLUMNS = {"observationStartMJD", "band", "filter"}

for column in common_columns:
    if column in SKIP_COLUMNS:
        continue
    if not (pd.api.types.is_numeric_dtype(df1[column]) and pd.api.types.is_numeric_dtype(df2[column])):
        continue
    fig, axes = plot_column_vs_mjd_compare(df1, df2, column, tagname1, tagname2)
    # too many points saturate the figure -> not saved by default
    # figname = f"compare_{tagname1}_vs_{tagname2}_{column}"
    # savefig(fig, figname, dpi=300)
    plt.show()